**1. Import**

In [34]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import io

**2.Load Dataset**

In [35]:
def load_dataset(file_path):
    df = pd.read_csv(file_path, sep=',')

    # Rename columns
    df.columns = ['user_id', 'item_id', 'normalized_rating']

    # Normalize the rating
    df['normalized_rating'] = df['normalized_rating'] / 10

    # Display first few rows
    df.head()
    return df

**3. Item Popularity**

In [36]:
def compute_item_popularity(df):
    return df["item_id"].value_counts(normalize=True)

**4. Spammer Simulation**

In [37]:
def simulate_random_spammers(num_spammers, item_popularity_dist, user_id_start, lambda_poisson=5):
    spam_data = []
    item_ids = item_popularity_dist.index.tolist()
    item_probs = item_popularity_dist.values
    max_rating = 5

    for i in range(num_spammers):
        user_id = user_id_start + i
        num_ratings = np.random.poisson(lam=lambda_poisson) + 1
        sampled_items = np.random.choice(item_ids, size=num_ratings, replace=False, p=item_probs)

        for movie_id in sampled_items:
            rating = np.random.randint(1, max_rating + 1)
            normalized = (rating - 1) / (max_rating - 1)
            spam_data.append([user_id, movie_id, normalized])


    return pd.DataFrame(spam_data, columns=['user_id', 'item_id', 'normalized_rating'])

**5. Combine Real Data + Spam**

In [38]:
def add_spammers_to_dataset(df, spammer_ratio=0.1, lambda_poisson=5):
    total_users = df["user_id"].nunique()
    num_spammers = int(np.ceil(spammer_ratio * total_users))
    item_popularity = compute_item_popularity(df)
    user_id_start = df["user_id"].max() + 1
    spam_df = simulate_random_spammers(num_spammers, item_popularity, user_id_start, lambda_poisson)
    return pd.concat([df, spam_df], ignore_index=True), spam_df


**6. Generate Multiple CSV's**

In [39]:
def generate_spam_datasets(base_df, ratios, lambda_poisson=5, output_dir="spam_versions"):
    os.makedirs(output_dir, exist_ok=True)
    for ratio in ratios:
        combined_df, spam_df = add_spammers_to_dataset(base_df, spammer_ratio=ratio, lambda_poisson=lambda_poisson)
        percent = int(ratio * 100)
        combined_df.to_csv(f"/home/martimsbaltazar/Desktop/tese/datasets/BookCrossing/{output_dir}/ratings_with_{percent}percent_spam.csv", index=False)
        spam_df.to_csv(f"/home/martimsbaltazar/Desktop/tese/datasets/BookCrossing/{output_dir}/spam_only_{percent}percent.csv", index=False)
        print(f"✔ Generated {percent}% spam version with {len(spam_df)} fake ratings.")

**7. Main Function**

In [40]:
if __name__ == "__main__":
    df_real = load_dataset("/home/martimsbaltazar/Desktop/tese/datasets/BookCrossing/Shortened_Ratings.csv")
    print(df_real.columns)
    spam_ratios = [0.10, 0.30, 0.50, 0.70]  
    generate_spam_datasets(df_real, spam_ratios, lambda_poisson=20)

Index(['user_id', 'item_id', 'normalized_rating'], dtype='object')
✔ Generated 10% spam version with 12462 fake ratings.
✔ Generated 30% spam version with 37512 fake ratings.
✔ Generated 50% spam version with 62839 fake ratings.
✔ Generated 70% spam version with 88032 fake ratings.
